# Day 10: Scaling RAG, Walked Through

Everything through Day 9 was tested at toy scale (6 documents). This notebook
shows what actually happens as that grows to thousands and beyond: latency,
indexing, parallelism, caching, sharding, query optimization, cost, and
monitoring.

Run each cell in order. Plain Python throughout, self-contained, and kept fast
enough to run end-to-end in under a minute.

## Cell 1: Why scaling matters

A RAG system tested only at small scale can fall apart in production: more
documents, more concurrent users, more data to store. Scaling isn't just "bigger
hardware" -- it's smarter design: indexing, caching, batching, sharding, and
monitoring.

## Cell 2: Show scaling benchmarks

Real latency numbers for brute-force search as the document count grows.

In [ ]:
import math, random, time

def make_random_vectors(count, dimensions=8, seed=42):
    rng = random.Random(seed)
    return [[rng.random() for _ in range(dimensions)] for _ in range(count)]

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    ma, mb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot / (ma * mb) if ma and mb else 0.0

def brute_force_search(query, vectors, top_k=5):
    scored = [(i, cosine_similarity(query, v)) for i, v in enumerate(vectors)]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]

for size in [100, 1_000, 10_000, 50_000]:
    vectors = make_random_vectors(size, seed=42)
    query = make_random_vectors(1, seed=999)[0]
    start = time.perf_counter()
    brute_force_search(query, vectors, top_k=5)
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"{size:>7,} documents: {elapsed_ms:>8.2f} ms per search")

## Cell 3: Impact of indexing

A simple partition index checks only one bucket instead of everything.

In [ ]:
class SimplePartitionIndex:
    def __init__(self, vectors, num_buckets=10, seed=42):
        self.vectors = vectors
        rng = random.Random(seed)
        anchor_indices = rng.sample(range(len(vectors)), min(num_buckets, len(vectors)))
        self.anchors = [vectors[i] for i in anchor_indices]
        self.buckets = [[] for _ in self.anchors]
        for i, v in enumerate(vectors):
            best = max(range(len(self.anchors)), key=lambda a: cosine_similarity(v, self.anchors[a]))
            self.buckets[best].append(i)
    def search(self, query, top_k=5):
        best = max(range(len(self.anchors)), key=lambda a: cosine_similarity(query, self.anchors[a]))
        candidates = self.buckets[best]
        scored = sorted(((i, cosine_similarity(query, self.vectors[i])) for i in candidates), key=lambda p: p[1], reverse=True)
        return scored[:top_k], len(candidates)

size = 20_000
vectors = make_random_vectors(size, seed=42)
query = make_random_vectors(1, seed=999)[0]

start = time.perf_counter()
brute_results = brute_force_search(query, vectors, top_k=5)
brute_ms = (time.perf_counter() - start) * 1000

index = SimplePartitionIndex(vectors, num_buckets=size // 200)
start = time.perf_counter()
indexed_results, checked = index.search(query, top_k=5)
indexed_ms = (time.perf_counter() - start) * 1000

print(f"Brute force:  {brute_ms:.3f} ms, checked {size:,}/{size:,}")
print(f"Indexed:      {indexed_ms:.3f} ms, checked {checked:,}/{size:,}")
print(f"Speedup: {brute_ms / indexed_ms:.1f}x")

## Cell 4: Batching and parallelization

Fixed per-call overhead (like an API round-trip) adds up fast one at a time.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def simulate_call(x, overhead_s=0.01):
    time.sleep(overhead_s)
    return x

items = list(range(30))

start = time.perf_counter()
[simulate_call(x) for x in items]
sequential_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as executor:
    list(executor.map(simulate_call, items))
parallel_ms = (time.perf_counter() - start) * 1000

print(f"Sequential: {sequential_ms:.1f} ms")
print(f"Parallel (6 workers): {parallel_ms:.1f} ms ({sequential_ms/parallel_ms:.1f}x faster)")

## Cell 5: Caching strategies

Real traffic has repeats -- caching turns a repeat request into a free one.

In [ ]:
class SimpleCache:
    def __init__(self, compute_fn):
        self.compute_fn, self.store, self.hits, self.misses = compute_fn, {}, 0, 0
    def get(self, key):
        if key in self.store:
            self.hits += 1
            return self.store[key]
        self.misses += 1
        self.store[key] = self.compute_fn(key)
        return self.store[key]

rng = random.Random(42)
popular = [f"q{i}" for i in range(4)]
rare = [f"q{i}" for i in range(4, 20)]
requests = [rng.choice(popular) if rng.random() < 0.7 else rng.choice(rare) for _ in range(40)]

start = time.perf_counter()
for r in requests:
    simulate_call(r, overhead_s=0.01)
uncached_ms = (time.perf_counter() - start) * 1000

cache = SimpleCache(lambda k: simulate_call(k, overhead_s=0.01))
start = time.perf_counter()
for r in requests:
    cache.get(r)
cached_ms = (time.perf_counter() - start) * 1000

print(f"Without caching: {uncached_ms:.0f} ms")
print(f"With caching:    {cached_ms:.0f} ms ({uncached_ms/cached_ms:.1f}x faster)")
print(f"Hit rate: {cache.hits / (cache.hits + cache.misses):.0%}")

## Cell 6: Sharding concepts

Splitting data across shards, then routing or scattering queries to find it.

In [ ]:
def shard_documents(vectors, num_shards):
    shards = [[] for _ in range(num_shards)]
    for i, v in enumerate(vectors):
        shards[i % num_shards].append(v)
    return shards

def scatter_gather(shards, query, top_k=5):
    all_results = []
    for shard in shards:
        all_results.extend(brute_force_search(query, shard, top_k=top_k))
    all_results.sort(key=lambda r: r[1], reverse=True)
    return all_results[:top_k]

shards = shard_documents(vectors, num_shards=4)
print(f"Split {len(vectors):,} documents into 4 shards of ~{len(shards[0]):,} each")
results = scatter_gather(shards, query, top_k=5)
print(f"Top result similarity after scatter-gather: {results[0][1]:.3f}")
print("(Each shard searched sequentially here -- in production, shards run in")
print(" PARALLEL across machines, which is where the real speedup comes from.)")

## Cell 7: Query optimization

Approximate search checks a sample instead of everything -- faster, with a
measurable quality cost.

In [ ]:
def approximate_search(vectors, query, top_k=5, sample_fraction=0.1, seed=42):
    rng = random.Random(seed)
    sample_size = max(top_k, int(len(vectors) * sample_fraction))
    sample = rng.sample(range(len(vectors)), sample_size)
    scored = sorted(((i, cosine_similarity(query, vectors[i])) for i in sample), key=lambda p: p[1], reverse=True)
    return scored[:top_k], sample_size

exact_results = brute_force_search(query, vectors, top_k=5)
approx_results, sample_size = approximate_search(vectors, query, top_k=5)

print(f"Exact search:       checked {len(vectors):,}, top score {exact_results[0][1]:.3f}")
print(f"Approximate (10%):  checked {sample_size:,}, top score {approx_results[0][1]:.3f}")
print(f"Quality gap: {exact_results[0][1] - approx_results[0][1]:+.3f}")

## Cell 8: Cost analysis

Embedding cost is one-time; storage and compute are recurring and grow with usage.

In [ ]:
def embedding_cost(n, avg_tokens=200, price_per_1k=0.00002):
    return (n * avg_tokens / 1000) * price_per_1k

def compute_cost_per_month(n, queries_per_month, price_per_query_ms=0.000001, latency_ms_per_1k=2.0):
    latency_per_query = (n / 1000) * latency_ms_per_1k
    return latency_per_query * queries_per_month * price_per_query_ms

for size in [1_000, 100_000, 10_000_000]:
    embed = embedding_cost(size)
    compute = compute_cost_per_month(size, queries_per_month=100_000)
    print(f"{size:>10,} docs: embed ${embed:>8.2f} (one-time), compute ${compute:>9.2f}/month")

## Cell 9: Monitoring

Averages hide problems that percentiles reveal.

In [ ]:
import statistics

def simulate_latencies(n, seed=42):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        out.append(rng.gauss(50, 10) if rng.random() < 0.95 else rng.gauss(800, 200))
    return [max(1.0, v) for v in out]

def percentile(values, p):
    s = sorted(values)
    return s[min(len(s) - 1, int(len(s) * p / 100))]

latencies = simulate_latencies(1000)
print(f"Average: {statistics.mean(latencies):.1f} ms")
print(f"p50:     {percentile(latencies, 50):.1f} ms")
print(f"p95:     {percentile(latencies, 95):.1f} ms")
print(f"p99:     {percentile(latencies, 99):.1f} ms  <- the average completely hides this")

## Cell 10: Key takeaways

- Scaling isn't just bigger hardware -- it's smarter design: indexing, caching,
  batching, sharding, and monitoring.
- Brute-force search latency grows roughly linearly with document count -- fine
  at small scale, a real problem well before a million documents.
- Indexing trades a small amount of accuracy for staying fast as data grows,
  checking a tiny fraction of the data instead of all of it.
- Batching and parallelism both fight the same enemy: fixed per-call overhead that
  adds up when paid one item at a time.
- Caching is a near-free win whenever real traffic has repeats -- which it almost
  always does.
- Sharding enables TRUE scale beyond one machine's capacity, at a real complexity
  cost -- reach for it once you've actually hit that wall, not by default.
- Approximate search trades a measurable, checkable quality cost for speed -- safe
  when a near-best answer is fine, risky when exactness has real consequences.
- Compute cost can grow faster than storage or embedding cost as both document
  count and query volume increase -- indexing isn't just a latency fix, it's a
  cost fix.
- Monitor percentiles, not averages -- the average can look perfectly healthy while
  a real, painful tail of slow requests goes completely unnoticed.